In [1]:
import os
import sys
import json
import random
from pathlib import Path
from typing import Any, Dict, Iterator, List, Optional

import torch
from torch.utils.data import IterableDataset
from datasets import load_dataset, interleave_datasets
from transformers import Trainer, TrainingArguments, set_seed

import sys
sys.path.insert(0, "../")

from src.model.gigachat_vl import GigaChatVL
from src.dataset.finevision import load_finevision_streaming, \
    FineVisionIterableDataset, VLMDataCollator
from src.utils.train_utils import save_artifacts

In [2]:
PROJECT_ROOT = "/media/alexey/SSDData/experiments/gigachat_vl/"
DATASET_ROOT = "/media/alexey/HDDLargeData/datasets/llm/VL/FineVision_mix"
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs", "gigachat_vl_finevision_local")

for p in [PROJECT_ROOT]:
    if p not in sys.path:
        sys.path.append(p)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# Path config to fix TypeError

# LLM_NAME = "/media/alexey/HDDLargeData/models/VLM/GigaChat3.1-10B-A1.8B-bf16"

# config_path = os.path.join(LLM_NAME, "config.json")

# with open(config_path, "r", encoding="utf-8") as f:
#     cfg = json.load(f)

# print("Before:", cfg.get("routed_scaling_factor"), type(cfg.get("routed_scaling_factor")))

# if "routed_scaling_factor" in cfg and isinstance(cfg["routed_scaling_factor"], int):
#     cfg["routed_scaling_factor"] = float(cfg["routed_scaling_factor"])

# with open(config_path, "w", encoding="utf-8") as f:
#     json.dump(cfg, f, ensure_ascii=False, indent=2)

# print("After:", cfg.get("routed_scaling_factor"), type(cfg.get("routed_scaling_factor")))
# print("Patched:", config_path)

In [3]:
LLM_PATH = "/media/alexey/HDDLargeData/models/VLM/GigaChat3.1-10B-A1.8B-bf16"
VISION_PATH = "/media/alexey/HDDLargeData/models/VLM/Qwen2.5-VL-7B-Instruct/"

FINEVISION_SUBSETS = ["chartqa", "docvqa", "textvqa"]

MAX_STEPS = 5000
LR = 2e-4
WEIGHT_DECAY = 0.0
WARMUP_RATIO = 0.03
warmup_steps = int(MAX_STEPS * WARMUP_RATIO)

TRAIN_BS = 1
GRAD_ACCUM = 16
MAX_LENGTH = 2048

SHUFFLE_BUFFER = 1000
LOGGING_STEPS = 10
SAVE_STEPS = 500
SEED = 42

USE_4BIT_LLM = True
FREEZE_VISION = True

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

SHUFFLE_CONVERSATIONS = False
SKIP_MULTI_IMAGE = True
MAX_TURNS_PER_ROW = None

set_seed(SEED)

model = GigaChatVL(
    llm_name=LLM_PATH,
    vision_name=VISION_PATH,
    use_4bit_llm=USE_4BIT_LLM,
    freeze_vision=FREEZE_VISION,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
)

if hasattr(model.llm, "gradient_checkpointing_enable"):
    try:
        model.llm.gradient_checkpointing_enable()
    except Exception as e:
        print(f"gradient_checkpointing_enable failed: {e}")

if hasattr(model.llm, "print_trainable_parameters"):
    try:
        model.llm.print_trainable_parameters()
    except Exception as e:
        print(f"print_trainable_parameters failed: {e}")

raw_stream = load_finevision_streaming(
    dataset_root=DATASET_ROOT,
    subsets=FINEVISION_SUBSETS,
    shuffle_buffer=SHUFFLE_BUFFER,
    seed=SEED,
)

train_dataset = FineVisionIterableDataset(
    raw_stream,
    shuffle_conversations=SHUFFLE_CONVERSATIONS,
    seed=SEED,
    skip_multi_image=SKIP_MULTI_IMAGE,
    max_turns_per_row=MAX_TURNS_PER_ROW,
)

collator = VLMDataCollator(
    model=model,
    max_length=MAX_LENGTH,
)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=TRAIN_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_strategy="steps",
    bf16=use_bf16,
    fp16=use_fp16,
    remove_unused_columns=False,
    report_to="none",
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collator,
)

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

DeepseekV3ForCausalLM LOAD REPORT from: /media/alexey/HDDLargeData/models/VLM/GigaChat3.1-10B-A1.8B-bf16
Key                                                 | Status     |  | 
----------------------------------------------------+------------+--+-
model.layers.26.shared_head.norm.weight             | UNEXPECTED |  | 
model.layers.26.self_attn.kv_a_proj_with_mqa.weight | UNEXPECTED |  | 
model.layers.26.input_layernorm.weight              | UNEXPECTED |  | 
model.layers.26.mlp.gate.weight                     | UNEXPECTED |  | 
model.layers.26.self_attn.kv_a_layernorm.weight     | UNEXPECTED |  | 
model.layers.26.self_attn.o_proj.weight             | UNEXPECTED |  | 
model.layers.26.self_attn.q_proj.weight             | UNEXPECTED |  | 
model.layers.26.shared_head.head.weight             | UNEXPECTED |  | 
model.layers.26.self_attn.kv_b_proj.weight          | UNEXPECTED |  | 
model.layers.26.enorm.weight                        | UNEXPECTED |  | 
model.layers.26.mlp.gate.e_score_correction

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

trainable params: 15,624,192 || all params: 10,687,414,784 || trainable%: 0.1462


Resolving data files:   0%|          | 0/25 [00:00<?, ?it/s]

In [ ]:
trainer.train()

Step,Training Loss


In [ ]:
save_artifacts(model, OUTPUT_DIR)
print(f"Saved final artifacts to: {OUTPUT_DIR}")